# 03 — Feature Engineering & Sequence Construction
**market-pulse-nn | ANN Project 1**

---

## Purpose
This notebook transforms the raw OHLCV price data and sentiment-labelled output
from Notebooks 01 & 02 into a **model-ready, normalised, windowed dataset** that
Notebook 04 (training) will consume directly.

**Run environment:** VS Code on Mac — CPU only, no GPU required (~2 min total).

---

## Pipeline covered here
```
sentiment_daily.csv  ─┐
                       ├─► Merge ─► Technical Indicators ─► Lag & Rolling Features
prices/TICKER.csv    ─┘         ─► Target Variable ─► Sliding Window ─► Scale ─► Save
```

---

## Outputs
| File | Location | Description |
|------|----------|-------------|
| `sequences.npz` | `data/final/` | All (X, y) arrays compressed |
| `scaler.pkl` | `data/final/` | Fitted StandardScaler (reuse in NB05) |
| `feature_cols.json` | `data/final/` | Ordered feature name list |
| `*.png` figures | `reports/figures/` | Saved visualisations for IEEE report |

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import sys, json, pickle, warnings
from pathlib import Path

# ── Numerical & data manipulation ─────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Technical analysis (ta-lib wrapper) ───────────────────────────────────────
# The `ta` library is a vectorised, no-look-ahead implementation of all standard
# technical indicators. It computes each indicator purely from past data.
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands, AverageTrueRange

# ── Scikit-learn preprocessing ────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Silence minor deprecation warnings that don't affect functionality
warnings.filterwarnings('ignore')

# Use a clean, publication-quality plot style.
# Try the renamed seaborn style first (matplotlib >= 3.6), fall back otherwise.
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')

# ── Project config ─────────────────────────────────────────────────────────────
# Insert the project root into sys.path so `config.py` is importable
# from within the notebooks/ sub-directory.
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from config import CFG

print(f'Environment  : {CFG.ENV}')
print(f'Project root : {CFG.ROOT}')
print(f'Tickers      : {CFG.TICKERS}')
print(f'Lookback     : {CFG.LOOKBACK} trading days')
print(f'Forecast     : {CFG.FORECAST_HORIZON} day(s) ahead')
print(f'Train/Val/Test split: {CFG.TRAIN_RATIO}/{CFG.VAL_RATIO}/{CFG.TEST_RATIO}')

---
## Step 1 — Load & Inspect Price Data

Each ticker has its own CSV produced by Notebook 01.
We load all five, confirm date ranges, and sort chronologically.
**Expected:** 1,003 trading days each, 2021-01-04 → 2024-12-27.

In [ ]:
price_dfs = {}

for ticker in CFG.TICKERS:
    path = CFG.PRICES / f'{ticker}_daily.csv'
    
    # 1. Read just the header first to check column names
    temp_cols = pd.read_csv(path, nrows=0).columns.tolist()
    
    # 2. Find the actual name of the date column (case-insensitive)
    # This looks for 'date', 'Date', 'DATE', etc.
    date_col = next((c for c in temp_cols if c.strip().lower() == 'date'), None)
    
    if date_col is None:
        raise ValueError(f"Could not find a date column in {path}. Found: {temp_cols}")

    # 3. Read the full CSV using the detected column name
    df = pd.read_csv(path, parse_dates=[date_col], index_col=date_col)
    
    # Standardize the index name to 'Date' for the rest of your script
    df.index.name = 'Date'
    
    df.sort_index(inplace=True) 
    df.index = pd.to_datetime(df.index).normalize()
    
    price_dfs[ticker] = df
    print(f'  {ticker:4s}  rows={len(df):,}  '
          f'range={df.index[0].date()} → {df.index[-1].date()}  '
          f'cols={df.columns.tolist()}')

print(f'\n{len(price_dfs)} tickers loaded.')

In [ ]:
# ── Visualise raw and normalised closing prices ───────────────────────────────
# Normalising all tickers to base 100 at the start of the period lets us compare
# their relative performance on a single scale regardless of absolute price level.
# NVDA's dramatic appreciation (~2 000% over this period) will be immediately visible.

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# ── Panel 1: Raw closing prices ──────────────────────────────────────────────
ax1 = axes[0]
for ticker, df in price_dfs.items():
    # Standardize column names to avoid KeyError
    df.columns = [c.strip().lower() for c in df.columns]
    
    # Use the lowercase 'close' now that we've renamed them
    ax1.plot(df.index, df['close'], lw=1.5, label=ticker)
    
ax1.set_title('Raw Closing Prices — All Tickers (2021–2024)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.legend(loc='upper left', framealpha=0.85)
ax1.grid(True, alpha=0.3)

# ── Panel 2: Normalised to base 100 ──────────────────────────────────────────
ax2 = axes[1]
for ticker, df in price_dfs.items():
    # Columns are already cleaned from the loop above if using the same objects
    norm = df['close'] / df['close'].iloc[0] * 100
    ax2.plot(df.index, norm, lw=1.5, label=ticker)

ax2.axhline(100, color='black', lw=0.8, ls='--', alpha=0.5, label='Base (100)')
ax2.set_title('Normalised Performance — Base 100', fontsize=13, fontweight='bold')
ax2.set_ylabel('Normalised Price')
ax2.set_xlabel('Date')
ax2.legend(loc='upper left', framealpha=0.85)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
# Ensure the directory exists before saving
CFG.FIGURES.mkdir(parents=True, exist_ok=True)
fig.savefig(CFG.FIGURES / '01_closing_prices.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 2 — Technical Indicators

Technical indicators compress complex price dynamics into interpretable scalar signals.
All indicators are **non-look-ahead** — each value at time *t* uses only data up to *t*.

| Indicator | Parameters | Captures |
|-----------|-----------|---------|
| **RSI** | window=14 | Overbought/oversold momentum; >70 = overbought, <30 = oversold |
| **MACD** | fast=12, slow=26, signal=9 | Trend direction and momentum crossovers |
| **Bollinger Bands** | window=20, std=2 | Volatility envelope; %B shows price position within bands |
| **ATR** | window=14 | Average daily price range — direct volatility proxy |

We also compute **daily return**, **log return**, and **volume change** as
direct price-action signals — these give the model a short-term momentum view
without the lag of a rolling indicator.

In [ ]:
def compute_technical_indicators(df: pd.DataFrame, cfg) -> pd.DataFrame:
    '''
    Adds RSI, MACD, Bollinger Bands, ATR, return, and volume features to a
    single-ticker OHLCV DataFrame in-place. Returns the enriched DataFrame.

    No look-ahead bias: every indicator value at time t uses only data
    available at t. The first N rows will contain NaN during warm-up
    (N = max indicator period = 26 for MACD slow window).
    '''
    # Standardize column names to lowercase to prevent KeyErrors
    df.columns = [c.strip().lower() for c in df.columns]

    # Map variables to the standardized lowercase column names
    close  = df['close']
    high   = df['high']
    low    = df['low']
    volume = df['volume']

    # ── RSI (Relative Strength Index) ─────────────────────────────────────────
    # RSI = 100 - 100 / (1 + avg_gain / avg_loss) over the lookback window.
    # Values 0–100. We pass the raw number rather than a binary threshold so
    # the model can learn any decision boundary that generalises to new data.
    df['rsi'] = RSIIndicator(close=close, window=cfg.RSI_PERIOD).rsi()

    # ── MACD (Moving Average Convergence Divergence) ───────────────────────────
    # MACD line  = EMA(fast) − EMA(slow)  — direction of the trend
    # Signal line = EMA(MACD, 9)           — smoothed MACD
    # Histogram   = MACD − signal          — momentum: positive = accelerating up
    macd_obj          = MACD(close=close, window_slow=cfg.MACD_SLOW,
                             window_fast=cfg.MACD_FAST, window_sign=cfg.MACD_SIGNAL)
    df['macd']        = macd_obj.macd()
    df['macd_signal'] = macd_obj.macd_signal()
    df['macd_hist']   = macd_obj.macd_diff()   # histogram = MACD − signal

    # ── Bollinger Bands ────────────────────────────────────────────────────────
    # Upper = SMA(20) + 2σ   Lower = SMA(20) − 2σ
    # %B = (price − lower) / (upper − lower)  ← where is price in the band?
    #   %B = 0  → price at lower band (oversold signal)
    #   %B = 1  → price at upper band (overbought signal)
    #   %B > 1 or < 0 → price outside band (breakout signal — high volatility)
    bb             = BollingerBands(close=close, window=cfg.BB_PERIOD, window_dev=cfg.BB_STD)
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_lower'] = bb.bollinger_lband()
    df['bb_pct']   = bb.bollinger_pband()  # %B — normalised position within band

    # ── ATR (Average True Range) ───────────────────────────────────────────────
    # True Range = max(H−L, |H−prev_C|, |L−prev_C|)
    # ATR = smoothed average of True Range over window days.
    # High ATR = high volatility. Used directly as the volatility spike predictor.
    df['atr'] = AverageTrueRange(
        high=high, low=low, close=close, window=cfg.ATR_PERIOD
    ).average_true_range()

    # ── Return features ────────────────────────────────────────────────────────
    # Simple return: (P_t − P_{t-1}) / P_{t-1}   — intuitive percentage move
    # Log return   : ln(P_t / P_{t-1})            — additive across time, preferred
    #                                                for financial time-series models
    df['daily_return'] = close.pct_change()
    df['log_return']   = np.log(close / close.shift(1))

    # ── Volume change ──────────────────────────────────────────────────────────
    # Percentage change in volume. Unusual volume often precedes large moves.
    # Using pct_change rather than raw volume makes the feature scale-invariant
    # across tickers with very different average trading volumes.
    df['volume_change'] = volume.pct_change()

    return df


# Apply the function to every ticker
print('Computing technical indicators for all tickers ...')
for ticker in CFG.TICKERS:
    price_dfs[ticker] = compute_technical_indicators(price_dfs[ticker], CFG)
    n_cols = len(price_dfs[ticker].columns)
    print(f'  {ticker:4s}  total columns now: {n_cols}')

print('\nNote: NaN values in the first ~26 rows per ticker are expected '
      '(indicator warm-up period). They will be dropped later.')

In [ ]:
# ── Visualise RSI, MACD, and Bollinger Bands for NVDA ────────────────────────
# NVDA is chosen as the illustrative ticker because its 2021–2024 price journey
# (bull → correction → explosive AI-driven rally) makes indicator signals vivid.
# This 3-panel chart is suitable for the IEEE report's methodology section.

ticker = 'NVDA'
df_plot = price_dfs[ticker].dropna().copy()

fig = plt.figure(figsize=(15, 11))
gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1.5, 1.5], hspace=0.06)

# ── Panel 1: Price + Bollinger Bands ─────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
# Changed 'Close' to 'close' to match lowercase standardization
ax1.plot(df_plot.index, df_plot['close'],    color='#2196F3', lw=1.8, label='Close', zorder=3)
ax1.plot(df_plot.index, df_plot['bb_upper'], color='#E91E63', lw=1.0, ls='--',
         label='BB Upper (2σ)', alpha=0.8)
ax1.plot(df_plot.index, df_plot['bb_lower'], color='#4CAF50', lw=1.0, ls='--',
         label='BB Lower (2σ)', alpha=0.8)
ax1.fill_between(df_plot.index, df_plot['bb_upper'], df_plot['bb_lower'],
                 alpha=0.07, color='#9E9E9E', label='Band region')
ax1.set_title(f'{ticker}  —  Closing Price + Bollinger Bands  (2021–2024)',
              fontsize=13, fontweight='bold', pad=10)
ax1.set_ylabel('Price (USD)', fontsize=10)
ax1.legend(loc='upper left', fontsize=9, framealpha=0.85)
ax1.grid(True, alpha=0.25)
ax1.tick_params(labelbottom=False)

# ── Panel 2: RSI ──────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.plot(df_plot.index, df_plot['rsi'], color='#7B1FA2', lw=1.3, label='RSI (14)')
ax2.axhline(70, color='#E91E63', lw=1.2, ls='--', alpha=0.8, label='Overbought (70)')
ax2.axhline(30, color='#4CAF50', lw=1.2, ls='--', alpha=0.8, label='Oversold (30)')
ax2.axhline(50, color='grey',    lw=0.8, ls=':',  alpha=0.5)
ax2.fill_between(df_plot.index, df_plot['rsi'], 70,
                 where=(df_plot['rsi'] > 70), alpha=0.18, color='#E91E63')
ax2.fill_between(df_plot.index, df_plot['rsi'], 30,
                 where=(df_plot['rsi'] < 30), alpha=0.18, color='#4CAF50')
ax2.set_ylim(0, 100)
ax2.set_ylabel('RSI', fontsize=10)
ax2.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax2.grid(True, alpha=0.25)
ax2.tick_params(labelbottom=False)

# ── Panel 3: MACD + Signal + Histogram ────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.plot(df_plot.index, df_plot['macd'],        color='#2196F3', lw=1.3, label='MACD')
ax3.plot(df_plot.index, df_plot['macd_signal'], color='#FF9800', lw=1.3, label='Signal')
# Colour histogram bars green (bullish) or red (bearish) based on sign
bar_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in df_plot['macd_hist']]
ax3.bar(df_plot.index, df_plot['macd_hist'],
        color=bar_colors, alpha=0.55, width=1.2, label='Histogram')
ax3.axhline(0, color='black', lw=0.8, alpha=0.4)
ax3.set_ylabel('MACD', fontsize=10)
ax3.set_xlabel('Date', fontsize=10)
ax3.legend(loc='upper left', fontsize=8, framealpha=0.85)
ax3.grid(True, alpha=0.25)

plt.tight_layout()
fig.savefig(CFG.FIGURES / '02_technical_indicators_NVDA.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/02_technical_indicators_NVDA.png')

In [ ]:
# ── Plot ATR (volatility) for all tickers ─────────────────────────────────────
# ATR spikes indicate high-volatility periods — exactly what the model should
# learn to predict. Notable events visible in ATR:
#   2022-Q1: Fed rate hike uncertainty (all tickers spike)
#   2022-Q4: TSLA sell-off (TSLA ATR dominates)
#   2023-Q4 → 2024: NVDA AI rally (sustained high ATR)

fig, ax = plt.subplots(figsize=(14, 5))

for ticker, df in price_dfs.items():
    df_clean = df.dropna()
    # Normalise ATR by closing price to make it percentage-of-price,
    # so tickers at different price levels are comparable on one axis.
    norm_atr = df_clean['atr'] / df_clean['close'] * 100
    ax.plot(df_clean.index, norm_atr, lw=1.2, alpha=0.8, label=ticker)

ax.set_title('ATR as % of Closing Price — All Tickers  (2021–2024)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('ATR / Close  (%)', fontsize=10)
ax.set_xlabel('Date', fontsize=10)
ax.legend(framealpha=0.85)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(CFG.FIGURES / '03_atr_all_tickers.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/03_atr_all_tickers.png')

---
## Step 3 — Sentiment Features

The sentiment data from Notebook 02 is daily-aggregated — one row per (trading day, ticker).
The primary column is `av_sentiment_mean`, derived from Alpha Vantage news articles.

**Why sparse coverage is OK:**
~88% of trading days have `av_sentiment_mean = 0` (no articles found for that ticker on
that day). This zero represents **neutral** — not missing — and is a valid signal.
Markets dominated by silence often continue their prior trend.

To amplify the sparse signal, we derive:

| Feature | Formula | Purpose |
|---------|---------|---------|
| `sent_roll_3d` | `rolling(3).mean()` | 3-day average — reduces noise |
| `sent_roll_5d` | `rolling(5).mean()` | 5-day average — smoother trend |
| `sent_lag_1/2/3` | `shift(1/2/3)` | Yesterday's / 2-days-ago / 3-days-ago sentiment |
| `sent_momentum` | `roll_3d − roll_5d` | Rate of change — captures sentiment reversals |

In [ ]:
# ── Load sentiment_daily.csv produced by Notebook 02 ─────────────────────────
# Format: one row per (trading date, ticker). Primary column: av_sentiment_mean.
# File was saved using Python's csv module (not pandas.to_csv) to work around
# a pyarrow incompatibility on Python 3.9 — see session handoff notes.

sent_path = CFG.DATA_PROC / 'sentiment_daily.csv'
sent_df   = pd.read_csv(sent_path, parse_dates=['date'])
sent_df['date'] = pd.to_datetime(sent_df['date']).dt.normalize()

print(f'Loaded: {sent_path.name}')
print(f'Shape : {sent_df.shape}   (rows = trading days × tickers)')
print(f'Columns: {sent_df.columns.tolist()}')
print()
print(sent_df.head(10).to_string())
print()

# ── Confirm the primary sentiment column exists ────────────────────────────────
# Alpha Vantage sentiment is stored in 'av_sentiment_mean'.
# If the column name differs slightly due to a notebook 02 variation,
# this block renames it to the expected name.
if 'av_sentiment_mean' not in sent_df.columns:
    # Try common alternatives
    candidates = [c for c in sent_df.columns if 'sentiment' in c.lower() and 'mean' in c.lower()]
    if candidates:
        sent_df.rename(columns={candidates[0]: 'av_sentiment_mean'}, inplace=True)
        print(f'  Renamed column: {candidates[0]} → av_sentiment_mean')
    else:
        raise ValueError(f'av_sentiment_mean not found. Available: {sent_df.columns.tolist()}')

print('\nSentiment coverage per ticker (% of days with non-zero score):')
for ticker in CFG.TICKERS:
    sub = sent_df[sent_df['ticker'] == ticker]
    nonzero_pct = (sub['av_sentiment_mean'] != 0).mean() * 100
    mean_score  = sub['av_sentiment_mean'].mean()
    print(f'  {ticker:4s}  coverage={nonzero_pct:5.1f}%   mean_score={mean_score:+.4f}')

In [ ]:
# ── Sentiment coverage visualisation ─────────────────────────────────────────
# Two panels:
#   1. Time-series of sentiment score per ticker — shows temporal patterns
#   2. Bar chart of non-zero coverage percentage per ticker

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Panel 1: Sentiment over time ──────────────────────────────────────────────
ax1 = axes[0]
palette = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0']
for i, ticker in enumerate(CFG.TICKERS):
    sub = sent_df[sent_df['ticker'] == ticker].set_index('date')['av_sentiment_mean']
    ax1.plot(sub.index, sub.values, lw=1.0, alpha=0.75,
             color=palette[i], label=ticker)
ax1.axhline(0,    color='black', lw=0.9, ls='--', alpha=0.5, label='Neutral (0)')
ax1.axhline(0.05, color='grey',  lw=0.6, ls=':',  alpha=0.4)
ax1.axhline(-0.05,color='grey',  lw=0.6, ls=':',  alpha=0.4)
ax1.set_title('Daily Sentiment Score — All Tickers  (av_sentiment_mean)',
              fontsize=13, fontweight='bold')
ax1.set_ylabel('Sentiment Score', fontsize=10)
ax1.legend(loc='upper left', fontsize=9, framealpha=0.85)
ax1.grid(True, alpha=0.3)

# ── Panel 2: Coverage bar chart ───────────────────────────────────────────────
ax2 = axes[1]
coverages = []
for ticker in CFG.TICKERS:
    sub = sent_df[sent_df['ticker'] == ticker]
    coverages.append((sub['av_sentiment_mean'] != 0).mean() * 100)

bars = ax2.bar(CFG.TICKERS, coverages, color=palette, edgecolor='white', lw=1.5)
for bar, val in zip(bars, coverages):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax2.set_title('Sentiment Coverage per Ticker — % of Trading Days with Articles',
              fontsize=13, fontweight='bold')
ax2.set_ylabel('Days with data (%)', fontsize=10)
ax2.set_ylim(0, max(coverages) * 1.35)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
fig.savefig(CFG.FIGURES / '04_sentiment_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/04_sentiment_coverage.png')

---
## Step 4 — Merge, Feature Engineering & Target Variable

For each ticker we:
1. **Left-join** the sentiment onto the price DataFrame (price is the master timeline)
2. Zero-fill any missing sentiment (no news = neutral)
3. Compute rolling sentiment and lag features
4. Compute price lag features
5. Create the binary target: `1` if next-day close > today's close, `0` otherwise

> **Data leakage check:** The target is created with `shift(-1)` (look forward), but
> the feature set uses only `shift(+k)` (look backward) and rolling windows that end
> at time *t*. There is **zero look-ahead bias** in the features.

In [ ]:
# ── Merge prices + sentiment and engineer all features ───────────────────────
all_ticker_dfs = {}

for ticker in CFG.TICKERS:
    df   = price_dfs[ticker].copy()

    # Extract only the date and sentiment score for this ticker
    sent = (
        sent_df[sent_df['ticker'] == ticker]
        .set_index('date')[['av_sentiment_mean']]
    )

    # Left join — price DataFrame is the authoritative time index.
    # Days without a sentiment entry become NaN, then filled with 0.
    df = df.join(sent, how='left')
    df['av_sentiment_mean'] = df['av_sentiment_mean'].fillna(0.0)
    # Zero-fill is intentional: absence of news = no new sentiment signal = neutral.

    # ── Rolling sentiment (smoothed) ──────────────────────────────────────────
    # Rolling means propagate the sparse signal across neighbouring days.
    # min_periods=1 ensures we get a value even at the start of the series
    # (rather than NaN during the first 2 or 4 days).
    df['sent_roll_3d'] = df['av_sentiment_mean'].rolling(3, min_periods=1).mean()
    df['sent_roll_5d'] = df['av_sentiment_mean'].rolling(5, min_periods=1).mean()

    # ── Sentiment lag features ────────────────────────────────────────────────
    # Lag features allow the model to detect delayed market reactions to news —
    # for instance, a positive article on Monday may lift prices on Wednesday.
    for lag in CFG.SENTIMENT_LAGS:   # [1, 2, 3]
        df[f'sent_lag_{lag}'] = df['av_sentiment_mean'].shift(lag)

    # ── Sentiment momentum ────────────────────────────────────────────────────
    # Difference between short-term and medium-term rolling sentiment.
    # Positive momentum = sentiment improving recently → potential bullish signal.
    df['sent_momentum'] = df['sent_roll_3d'] - df['sent_roll_5d']

    # ── Price lag features ────────────────────────────────────────────────────
    # Autoregressive component: give the model explicit access to recent prices
    # in addition to what the sequence window already provides.
    for lag in [1, 2, 3, 5]:         # subset of CFG.PRICE_LAGS for 27-feature target
        df[f'close_lag_{lag}'] = df['close'].shift(lag)

    all_ticker_dfs[ticker] = df

print('Feature engineering complete. Column summary for AAPL:')
sample = all_ticker_dfs['AAPL']
new_cols = [c for c in sample.columns if any(x in c for x in ['sent', 'lag', 'rsi', 'macd', 'bb', 'atr', 'return', 'volume_c'])]
print(f'  Engineered columns ({len(new_cols)}): {new_cols}')
print(f'  Total columns: {len(sample.columns)}')

In [ ]:
# ── Create the binary target variable: market direction ──────────────────────
# label = 1  →  PRICE GOES UP   (close[t+1] > close[t])
# label = 0  →  PRICE STAYS FLAT or GOES DOWN
#
# Implementation: shift close prices back by FORECAST_HORIZON (=1) and
# compare to today's close. shift(-1) means 'tomorrow's close'.
#
# The last FORECAST_HORIZON rows of each ticker will be NaN (no future data)
# and are removed with dropna().

for ticker in all_ticker_dfs:
    df = all_ticker_dfs[ticker]
    future_close  = df['close'].shift(-CFG.FORECAST_HORIZON)
    df['target']  = (future_close > df['close']).astype(int)
    # Remove rows where we don't have a future label
    all_ticker_dfs[ticker] = df.dropna(subset=['target'])

print(f'Target variable created: 1 = UP, 0 = DOWN/FLAT')
print(f'Forecast horizon: {CFG.FORECAST_HORIZON} day(s) ahead')
print()
print(f'{"Ticker":8s} {"UP":>6s} {"DOWN":>6s} {"UP %":>8s} {"Total":>8s}')
print('-' * 44)
for ticker, df in all_ticker_dfs.items():
    up   = int(df['target'].sum())
    down = len(df) - up
    print(f'{ticker:8s} {up:6d} {down:6d} {up/len(df)*100:7.1f}%  {len(df):7d}')

In [ ]:
# ── Visualise class balance ───────────────────────────────────────────────────
# Equity markets have a slight upward long-term bias, so we expect ~52–56% UP days.
# If imbalance is severe (>60/40), we would apply class weights in training.
# BCEWithLogitsLoss supports pos_weight parameter for this adjustment.

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Grouped bar chart ─────────────────────────────────────────────────────────
up_pcts   = [all_ticker_dfs[t]['target'].mean() * 100 for t in CFG.TICKERS]
down_pcts = [100 - p for p in up_pcts]
x = np.arange(len(CFG.TICKERS))
w = 0.38

b1 = axes[0].bar(x - w/2, up_pcts,   w, label='UP (1)',   color='#4CAF50', edgecolor='white', lw=1.2)
b2 = axes[0].bar(x + w/2, down_pcts, w, label='DOWN (0)', color='#F44336', edgecolor='white', lw=1.2)
axes[0].axhline(50, color='black', lw=1.2, ls='--', alpha=0.6, label='Balanced (50%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(CFG.TICKERS, fontsize=11)
axes[0].set_ylim(0, 75)
axes[0].set_title('Class Distribution per Ticker', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Percentage of Days (%)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')
for bar in b1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                 f'{bar.get_height():.1f}%', ha='center', fontsize=8, fontweight='bold')

# ── Pie chart: overall balance ────────────────────────────────────────────────
all_targets = pd.concat([df['target'] for df in all_ticker_dfs.values()])
total_up    = int(all_targets.sum())
total_down  = len(all_targets) - total_up
axes[1].pie(
    [total_up, total_down],
    labels=[f'UP  ({total_up:,})', f'DOWN  ({total_down:,})'],
    colors=['#4CAF50', '#F44336'],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    textprops={'fontsize': 11},
)
axes[1].set_title('Overall Class Balance — All Tickers Combined',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
fig.savefig(CFG.FIGURES / '05_class_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/05_class_balance.png')

---
## Step 5 — Feature Selection & Correlation Analysis

We define the canonical ordered list of **27 features** that will form each
timestep vector fed to the RNN/LSTM/GRU.

A correlation heatmap identifies redundant pairs (|r| > 0.97). Bollinger Band
upper/lower bounds are expected to be highly correlated with `Close` by construction,
but this is fine — recurrent layers learn to down-weight redundant signals.
What we verify is that sentiment features are largely **orthogonal** to price features,
confirming they carry independent predictive information.

In [ ]:
# ── Define the canonical 27-feature column set ────────────────────────────────
# Order matters: consistent ordering across all tickers and splits ensures the
# model always receives the same feature in the same input dimension.
# Groups are organised from market structure (OHLCV) → derived → sentiment.

FEATURE_COLS = [
    # ── Raw market data ───────────────────────────────────────────────────────
    'Open', 'High', 'Low', 'close', 'Volume',       # 5 features

    # ── Return signals (short-term price momentum) ────────────────────────────
    'daily_return', 'log_return',                    # 2 features

    # ── Momentum indicators ───────────────────────────────────────────────────
    'rsi',                                           # 1 feature
    'macd', 'macd_signal', 'macd_hist',              # 3 features

    # ── Volatility & band indicators ──────────────────────────────────────────
    'bb_upper', 'bb_lower', 'bb_pct',                # 3 features
    'atr',                                           # 1 feature
    'volume_change',                                 # 1 feature

    # ── Price lag features (autoregressive component) ─────────────────────────
    'close_lag_1', 'close_lag_2', 'close_lag_3', 'close_lag_5',   # 4 features

    # ── Sentiment features ────────────────────────────────────────────────────
    'av_sentiment_mean',                             # 1 feature (raw daily score)
    'sent_roll_3d', 'sent_roll_5d',                  # 2 features (smoothed)
    'sent_lag_1', 'sent_lag_2', 'sent_lag_3',        # 3 features (delayed reaction)
    'sent_momentum',                                 # 1 feature (sentiment direction)
]
# Total: 5 + 2 + 1 + 3 + 3 + 1 + 1 + 4 + 1 + 2 + 3 + 1 = 27

assert len(FEATURE_COLS) == 27, f'Expected 27 features, got {len(FEATURE_COLS)}'
print(f'Feature count: {len(FEATURE_COLS)}  ✓')
print()
for i, col in enumerate(FEATURE_COLS):
    print(f'  [{i+1:02d}] {col}')

In [ ]:
# ── Drop NaN rows and produce the correlation heatmap ───────────────────────

# FIX: Ensure FEATURE_COLS matches the lowercase standardization from previous steps
FEATURE_COLS = [c.lower() for c in FEATURE_COLS]

clean_dfs = {}
print('Dropping warm-up / lag NaN rows:')
for ticker, df in all_ticker_dfs.items():
    # We combine the lowercase features with the 'target' column
    df_clean = df[FEATURE_COLS + ['target']].dropna().copy()
    clean_dfs[ticker] = df_clean
    dropped = len(df) - len(df_clean)
    print(f'  {ticker:4s}  before={len(df):,}  after={len(df_clean):,}  dropped={dropped}')

# Verify no NaNs remain
total_nan = sum(df.isnull().sum().sum() for df in clean_dfs.values())
print(f'\nTotal remaining NaN values: {total_nan}   (should be 0)')

# ── Correlation heatmap (AAPL as representative ticker) ───────────────────────
# Pearson correlation between all 27 features on the full (NaN-free) AAPL dataset.
corr = clean_dfs['AAPL'][FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))  # mask upper triangle

fig, ax = plt.subplots(figsize=(17, 15))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',       # red = strong positive, blue = strong negative
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.4,
    linecolor='white',
    annot_kws={'size': 6.5},
    cbar_kws={'shrink': 0.7},
    ax=ax,
)
ax.set_title('Feature Correlation Matrix — AAPL (Pearson r)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.tight_layout()
fig.savefig(CFG.FIGURES / '06_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/06_feature_correlation.png')

---
## Step 6 — Sliding Window Sequence Construction

Sequential models require fixed-length **windows** rather than flat vectors.
We slide a window of `LOOKBACK = 60` trading days across each ticker's timeline:

```
Window i :  [day  1 … day 60]  →  label = direction at day 61
Window i+1: [day  2 … day 61]  →  label = direction at day 62
Window i+2: [day  3 … day 62]  →  label = direction at day 63
                       ...
```

**Output shape per ticker:** `(N_windows, 60, 27)`  
**Label shape:**              `(N_windows,)`

All 5 tickers are concatenated, then sorted chronologically, then split:
```
| ─────────────── Train (70%) ────────────── | ─ Val (15%) ─ | ─ Test (15%) ─ |
```

> Validation and test windows remain in chronological order to simulate
> real deployment. Only the training set is shuffled (after the split).

In [ ]:
def build_sequences(df: pd.DataFrame, lookback: int, feature_cols: list) -> tuple:
    '''
    Convert a flat, cleaned daily DataFrame into sliding-window sequences.

    Args:
        df:           Single-ticker DataFrame with feature_cols + target column.
        lookback:     Number of past timesteps per window (e.g. 60 = 3 months).
        feature_cols: Ordered list of 27 feature column names.

    Returns:
        X     : np.ndarray  shape (N, lookback, F)  — feature sequences
        y     : np.ndarray  shape (N,)              — binary labels
        dates : pd.DatetimeIndex                    — target date for each sample
    '''
    data    = df[feature_cols].values.astype(np.float32)  # (T, F)
    targets = df['target'].values.astype(np.float32)      # (T,)
    idx     = df.index                                     # DatetimeIndex

    X_list, y_list, d_list = [], [], []

    # Stride = 1: each window advances by one day. This maximises the number
    # of training samples and ensures the model sees every possible subsequence.
    for i in range(lookback, len(data)):
        X_list.append(data[i - lookback : i])  # (lookback, F) — past window
        y_list.append(targets[i])               # scalar label at day i
        d_list.append(idx[i])                   # the TARGET date (not window start)

    return (
        np.array(X_list, dtype=np.float32),     # (N, 60, 27)
        np.array(y_list, dtype=np.float32),     # (N,)
        pd.DatetimeIndex(d_list),
    )


# ── Apply to all tickers and concatenate ─────────────────────────────────────
print('Building sliding-window sequences (LOOKBACK=60, STRIDE=1) ...')
all_X, all_y, all_dates = [], [], []

for ticker, df in clean_dfs.items():
    X, y, dates = build_sequences(df, CFG.LOOKBACK, FEATURE_COLS)
    all_X.append(X)
    all_y.append(y)
    all_dates.append(dates)
    print(f'  {ticker:4s}  sequences={len(y):,}  '
          f'date range: {dates[0].date()} → {dates[-1].date()}')

# Stack all tickers into one combined dataset
X_all     = np.concatenate(all_X,     axis=0)   # (N_total, 60, 27)
y_all     = np.concatenate(all_y,     axis=0)   # (N_total,)
dates_all = np.concatenate(all_dates, axis=0)

print(f'\nCombined  X: {X_all.shape}   y: {y_all.shape}')

In [ ]:
# ── Feature normalisation with StandardScaler ─────────────────────────────────
# Neural networks converge significantly faster when all input features are on
# a similar scale. Without normalisation, features like Close (~$300) would
# produce gradients 60× larger than RSI (~50) or sentiment (~0.1), causing
# those large-scale features to dominate the weight updates.
#
# StandardScaler: x' = (x - mean) / std   →   zero mean, unit variance per feature
#
# CRITICAL — fit ONLY on the training set:
# If we fit on all data, the scaler would see future mean/std values, leaking
# information from val/test into the training distribution. We apply (transform)
# val and test using the training statistics.

N_train, T, F = X_train.shape   # (sequences, timesteps, features)

scaler = StandardScaler()

# sklearn expects 2D input: reshape (N, T, F) → (N*T, F), fit, then reshape back
X_train_2d = X_train.reshape(-1, F)
scaler.fit(X_train_2d)          # learns mean_ and scale_ from TRAINING data only

X_train = scaler.transform(X_train.reshape(-1, F)).reshape(N_train,       T, F)
X_val   = scaler.transform(X_val.reshape(-1, F)).reshape(len(y_val),   T, F)
X_test  = scaler.transform(X_test.reshape(-1, F)).reshape(len(y_test), T, F)

print('Normalisation complete (fit on train only).')
print(f'  Train mean (feature 0): {X_train[:,:,0].mean():.6f}  ← should be ≈ 0.0')
print(f'  Train std  (feature 0): {X_train[:,:,0].std():.6f}   ← should be ≈ 1.0')
print(f'  Global min after scale : {X_train.min():.3f}')
print(f'  Global max after scale : {X_train.max():.3f}')
print(f'  Val range   : [{X_val.min():.3f}, {X_val.max():.3f}]')
print(f'  Test range  : [{X_test.min():.3f}, {X_test.max():.3f}]')

In [ ]:
# ── Visualise the chronological split structure ────────────────────────────────
# Panel 1: horizontal timeline bar confirms no temporal overlap between splits.
# Panel 2: class balance per split — should be similar across train/val/test.
# If the test set has very different class balance from training, the model
# accuracy estimate may be misleading.

train_dates = d_sorted[:n_train]
val_dates   = d_sorted[n_train : n_train+n_val]
test_dates  = d_sorted[n_train+n_val:]

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# ── Panel 1: Timeline bar ─────────────────────────────────────────────────────
ax1 = axes[0]
t0       = pd.Timestamp(train_dates[0])
segments = [
    (train_dates[0],  train_dates[-1],  f'Train  ({len(y_train):,} sequences)', '#2196F3'),
    (val_dates[0],    val_dates[-1],    f'Val    ({len(y_val):,} sequences)',   '#FF9800'),
    (test_dates[0],   test_dates[-1],   f'Test   ({len(y_test):,} sequences)',  '#F44336'),
]
y_pos = 0
for start, end, label, color in segments:
    left  = (pd.Timestamp(start) - t0).days
    width = (pd.Timestamp(end)   - pd.Timestamp(start)).days
    ax1.barh(y_pos, width, left=left, height=0.45, color=color, label=label, alpha=0.9)
    ax1.text(left + width/2, y_pos, label, ha='center', va='center',
             fontsize=10, fontweight='bold', color='white')

ax1.set_yticks([])
ax1.set_xlabel('Days from dataset start  (2021-01-04)', fontsize=10)
ax1.set_title('Chronological Train / Validation / Test Split  — No Temporal Overlap',
              fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# ── Panel 2: Class balance per split ──────────────────────────────────────────
ax2 = axes[1]
split_names  = ['Train', 'Validation', 'Test']
split_labels = [y_train, y_val, y_test]
up_vals      = [y.mean() * 100 for y in split_labels]
dn_vals      = [100 - v for v in up_vals]

x = np.arange(3)
ax2.bar(x - 0.2, up_vals, 0.38, label='UP (1)',   color='#4CAF50', edgecolor='white', lw=1.2)
ax2.bar(x + 0.2, dn_vals, 0.38, label='DOWN (0)', color='#F44336', edgecolor='white', lw=1.2)
ax2.axhline(50, color='black', lw=1.2, ls='--', alpha=0.5, label='50% balanced')
ax2.set_xticks(x)
ax2.set_xticklabels(split_names, fontsize=11)
ax2.set_ylim(0, 70)
ax2.set_title('Class Balance per Split — Consistent Distribution Confirms No Split Leakage',
              fontsize=13, fontweight='bold')
ax2.set_ylabel('% of Sequences')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
for i, (u, d) in enumerate(zip(up_vals, dn_vals)):
    ax2.text(i - 0.2, u + 0.8, f'{u:.1f}%', ha='center', fontsize=9, fontweight='bold')
    ax2.text(i + 0.2, d + 0.8, f'{d:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
fig.savefig(CFG.FIGURES / '07_split_visualisation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → reports/figures/07_split_visualisation.png')

In [ ]:
# ── Save all outputs consumed by downstream notebooks ────────────────────────
#
# sequences.npz — compressed numpy archive containing all 6 arrays.
#   Load in NB04: data = np.load(path); X_train = data['X_train']
#
# scaler.pkl    — fitted StandardScaler. MUST be reused in NB05 for evaluation.
#   Any probability outputs from the model are in scaled feature space;
#   the scaler is needed for interpretability / inverse transforms.
#
# feature_cols.json — ordered list of 27 feature names. Used in NB05 for
#   per-feature importance analysis and IEEE report tables.

# ── sequences.npz ─────────────────────────────────────────────────────────────
npz_path = CFG.DATA_FINAL / 'sequences.npz'
np.savez_compressed(
    npz_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val,     y_val=y_val,
    X_test=X_test,   y_test=y_test,
)
size_mb = npz_path.stat().st_size / 1e6
print(f'[1/3]  sequences.npz   saved  ({size_mb:.1f} MB)  →  {npz_path}')

# ── scaler.pkl ────────────────────────────────────────────────────────────────
scaler_path = CFG.DATA_FINAL / 'scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'[2/3]  scaler.pkl      saved           →  {scaler_path}')

# ── feature_cols.json ─────────────────────────────────────────────────────────
cols_path = CFG.DATA_FINAL / 'feature_cols.json'
with open(cols_path, 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f'[3/3]  feature_cols.json saved         →  {cols_path}')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=' * 65)
print('  Notebook 03 — Feature Engineering & Sequence Construction')
print('  STATUS: COMPLETE')
print('=' * 65)
print()
print(f'  Tickers             : {CFG.TICKERS}')
print(f'  Features per step   : {len(FEATURE_COLS)}')
print(f'  Lookback window     : {CFG.LOOKBACK} trading days (~3 months)')
print(f'  Forecast horizon    : {CFG.FORECAST_HORIZON} day ahead')
print()
print('  Dataset shapes:')
print(f'    X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'    X_val   : {X_val.shape}    y_val   : {y_val.shape}')
print(f'    X_test  : {X_test.shape}    y_test  : {y_test.shape}')
print()
print('  Outputs saved to data/final/:')
print('    sequences.npz   — training arrays')
print('    scaler.pkl      — StandardScaler (reuse in NB05)')
print('    feature_cols.json')
print()
print('  Figures saved to reports/figures/:')
for i, name in enumerate(['01_closing_prices', '02_technical_indicators_NVDA',
                           '03_atr_all_tickers', '04_sentiment_coverage',
                           '05_class_balance', '06_feature_correlation',
                           '07_split_visualisation'], 1):
    print(f'    {i}. {name}.png')
print()
print('  Next: Upload project folder to Google Drive,')
print('        then run 04_model_training.ipynb on Colab A100.')
print('=' * 65)